In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences #type: ignore
from tensorflow.keras.models import Sequential #type: ignore
from tensorflow.keras.layers import Dense, Embedding, LSTM, Bidirectional, Dropout, SpatialDropout1D #type: ignore
from tensorflow.keras.metrics import AUC #type: ignore
from tensorflow.keras.callbacks import EarlyStopping # type: ignore
from sklearn.metrics import roc_auc_score 

In [2]:
train_df = pd.read_csv("train.csv/train.csv")

In [3]:
train_df.columns

Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='str')

In [4]:
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

X = train_df['comment_text'].values
y = train_df[labels].values

In [5]:
X, y

(<StringArray>
 [                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   'Explanation\nWhy the edits made under my username Hardcore Metallica Fan were reverted? They weren'

In [6]:
lengths = [len(seq) for seq in train_df['comment_text']]
max_length = max(lengths)

print("Max length:", max_length)

Max length: 5000


In [7]:
import numpy as np

print(np.percentile(lengths, 50)) 
print(np.percentile(lengths, 90))
print(np.percentile(lengths, 95))
print(np.percentile(lengths, 99))

205.0
889.0
1355.0
3444.0


In [8]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X)

vocab_size = len(tokenizer.word_index)
print("Total unique words:", vocab_size)

Total unique words: 210337


In [9]:
word_counts = tokenizer.word_counts
sorted_counts = sorted(word_counts.values(), reverse=True)

import numpy as np
coverage = np.cumsum(sorted_counts) / sum(sorted_counts)

print("Top 20k coverage:", coverage[19999])

Top 20k coverage: 0.9588574839447873


In [10]:
# since 20k words cover almost 96% of the text, refitting the tokenizer on X with max_word limit as 20k
max_words = 20000
tokenizer = Tokenizer(num_words=max_words, lower=True)
tokenizer.fit_on_texts(X)

In [11]:
# converting to sequences, max_sequence length is 1400 as it covers almost 95% of data

X_sequences = tokenizer.texts_to_sequences(X)
max_length = 1400
X_padded = pad_sequences(
    X_sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print("Final shape: ", X_padded.shape)

Final shape:  (159571, 1400)


In [12]:
from tensorflow.keras.models import Sequential #type: ignore
from tensorflow.keras.layers import Dense, Embedding, LSTM, Bidirectional, Dropout #type: ignore
from tensorflow.keras.metrics import AUC #type: ignore

In [ ]:
model = Sequential()

model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_length))
model.add(SpatialDropout1D(0.2))

model.add(Bidirectional(LSTM(64, return_sequences=True)))
model.add(Dropout(0.2))

model.add(Bidirectional(LSTM(64, return_sequences=False)))

model.add(Dropout(0.3))
model.add(Dense(6, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=[AUC(multi_label=True, name='auc')]
)

# Early Stopping
early_stop = EarlyStopping(
    monitor='val_auc', 
    mode='max', 
    patience=2, 
    restore_best_weights=True
)

In [18]:
history_lstm = model.fit(
    X_padded, y, 
    epochs=15, 
    batch_size=128, 
    validation_split=0.2,
    callbacks=[early_stop]
)

# Saving Model
model.save("trained_model_1.keras")
print("\nModel saved successfully as 'trained_model_1.keras'")

Epoch 1/15
998/998 ━━━━━━━━━━━━━━━━━━━━ 556s 555ms/step - auc: 0.9098 - loss: 0.0775 - val_auc: 0.9586 - val_loss: 0.0494
Epoch 2/15
998/998 ━━━━━━━━━━━━━━━━━━━━ 554s 556ms/step - auc: 0.9666 - loss: 0.0473 - val_auc: 0.9657 - val_loss: 0.0477
Epoch 3/15
998/998 ━━━━━━━━━━━━━━━━━━━━ 567s 568ms/step - auc: 0.9724 - loss: 0.0432 - val_auc: 0.9658 - val_loss: 0.0477
Epoch 4/15
998/998 ━━━━━━━━━━━━━━━━━━━━ 566s 567ms/step - auc: 0.9751 - loss: 0.0395 - val_auc: 0.9634 - val_loss: 0.0479
Epoch 5/15
998/998 ━━━━━━━━━━━━━━━━━━━━ 568s 569ms/step - auc: 0.9804 - loss: 0.0360 - val_auc: 0.9570 - val_loss: 0.0501

Model saved successfully as 'trained_model_1.keras'


In [19]:
X_test = pd.read_csv("test.csv/test.csv")
y_test = pd.read_csv("test_labels.csv/test_labels.csv")

In [20]:
X_test_sequences = tokenizer.texts_to_sequences(X_test['comment_text'])

X_test_padded = pad_sequences(
    X_test_sequences,
    maxlen=max_length,  # same as training
    padding='post',
    truncating='post'
)

if 'id' in y_test.columns:
    y_true = y_test.drop(columns=['id']).values
else:
    y_true = y_test.values

In [21]:
valid_rows = (y_true != -1).all(axis=1)

X_test_padded = X_test_padded[valid_rows]
y_true = y_true[valid_rows]

In [22]:
loss, auc = model.evaluate(X_test_padded, y_true, verbose=1)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test AUC: {auc:.4f}")

y_pred = model.predict(X_test_padded)


2000/2000 ━━━━━━━━━━━━━━━━━━━━ 469s 235ms/step - auc: 0.9606 - loss: 0.0757

Test Loss: 0.0757
Test AUC: 0.9606
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 430s 215ms/step


In [23]:
macro_auc = roc_auc_score(y_true, y_pred, average='macro')
weighted_auc = roc_auc_score(y_true, y_pred, average='weighted')

print(f"\nMacro AUC: {macro_auc:.4f}")
print(f"Weighted AUC: {weighted_auc:.4f}")

print("\nPer-label AUC:")
label_names = y_test.columns.tolist()
if 'id' in label_names:
    label_names.remove('id')

for i, label in enumerate(label_names):
    auc_score = roc_auc_score(y_true[:, i], y_pred[:, i])
    print(f"{label}: {auc_score:.4f}")


Macro AUC: 0.9668
Weighted AUC: 0.9677

Per-label AUC:
toxic: 0.9642
severe_toxic: 0.9871
obscene: 0.9762
threat: 0.9580
insult: 0.9673
identity_hate: 0.9478
